# Kütüphaneler

In [1]:
import pandas as pd
import numpy as np
import pandas.io.formats.excel as pife
import warnings
import pickle
import os
from termcolor import colored
from pandas.errors import SettingWithCopyWarning
from thefuzz import fuzz

# Ayarlar

In [2]:
os.system('color')
pd.options.display.float_format = '{:,.2f}'.format
pife.ExcelFormatter.header_style = None

warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = pd.errors.ParserWarning)
warnings.filterwarnings('ignore', category = SettingWithCopyWarning)

base_path = 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\'

pickle_folder_path = base_path + 'pickle' + '\\'
data_folder_path = base_path + 'data' + '\\'
output_folder_path = base_path + 'output' + '\\'

    
# mnvt_file_name = 'Müşteri No - VKN - TCKN.csv'
# test_file_name = 'Mizan Test.xlsx'
# test_file_name_2 = 'Mizan Test 2.xlsx'
# test_file_name_3 = 'Mizan Test 3.xlsx'
test_file_name = 'Mizan Test Prod.xlsx'

# Yardımcı Fonksiyonlar

## Tablo Manipülasyonu

In [3]:
def add_h_space(df_original, space=1):
    df_final = df_original.copy()
    width = df_final.shape[1]
    for _ in range(space):
        df_final.loc[len(df_final)] = [np.nan] * width
    return df_final

def add_v_space(df_original, space=1):
    df_final = df_original.copy()
    df_null = pd.DataFrame(columns=[np.nan] * space)
    df_final = pd.concat([df_final, df_null], axis=1)
    return df_final
    

def h_stack(df_list, space=3):
    df_first = df_list[0].copy().reset_index(drop=True)
    df_list_new = [df_first]
    df_null = pd.DataFrame({np.nan: [np.nan]})
    for df in df_list[1:]:
        df_temp = df.copy()
        for _ in range(space):
            df_list_new.append(df_null)
        df_list_new.append(df_temp.reset_index(drop=True))
    df_final = pd.concat(df_list_new, axis=1).reset_index(drop=True)
    return df_final


def v_stack(df_list, space=3):
    max_width = max([x.shape[1] for x in df_list])

    df_first = df_list[0].copy().reset_index(drop=True)
    if df_first.shape[1] < max_width:
        df_first = add_v_space(df_first, max_width - df_first.shape[1])
    df_list_new = [df_first.copy()]
    
    df_null = pd.DataFrame(columns=df_first.columns)
    df_null = add_h_space(df_null, space)

    for df in df_list[1:]:
        df_temp = df.copy()
        if df_temp.shape[1] < max_width:
            df_temp = add_v_space(df, max_width - df_temp.shape[1])

        df_new = pd.DataFrame([df_temp.columns], columns=df_first.columns)
        df_temp.columns = df_first.columns

        df_list_new.append(df_null)
        df_list_new.append(df_new)
        df_list_new.append(df_temp.reset_index(drop=True))

    df_final = pd.concat(df_list_new).reset_index(drop=True)
    return df_final


def insert_header(df_original, column):
    df_final = df_original.copy()
    header = [column] if type(column) is str else column
    df_final = v_stack([pd.DataFrame(columns=header), df_final], 0)
    return df_final

def insert_corner(df_original):
    df_final = df_original.copy()
    df_final = h_stack([pd.DataFrame(), v_stack([pd.DataFrame(), df_final], 0)], 1)
    return df_final

def insert_row(df_original, index=0, row=None, count=1):
    df_final = df_original.copy().reset_index(drop=True)
    
    if row is None:
        row = np.nan
    if type(row) is list and len(row) < df_final.shape[1]:
        row += [np.nan] * (df_final.shape[1] - len(row))
    if index < 0:
        index += df_final.shape[0]

    new_indices = [x for x in range(index)] + [x + count for x in range(index, len(df_final))]
    df_final.index = new_indices

    for i in range(count):
        df_final.loc[index + i] = row

    df_final = df_final.reset_index().sort_values('index').drop('index', axis=1).reset_index(drop=True)
    return df_final

## Verim Tabloları

In [4]:
def fix_verim_table(df, columns=None):
    start = 0
    if columns is None:
        start = 3
        columns = df.iloc[2, 1:]
    df = df.iloc[start:, 1:]
    df.columns = columns
    df.columns.name = None
    df = df.reset_index(drop=True)
    return df

def quick_export_verim_csv(data, output_file_name, sep=';', header=False, index=False, open_file=False):
    cl = [str(x) for x in range(0, 16)]
    df = pd.DataFrame()
    df['10'] = data
    for c in ['0', '15']:
        df[c] = 'x'
    for c in cl:
        if c not in ['0', '10', '15']:
            df[c] = np.nan
    df = df[cl]

    output_file_path = output_folder_path + output_file_name + '.csv'
    df.to_csv(output_file_path, sep=sep, header=header, index=index)

    if open_file:
        os.startfile(output_file_path)
    

## Hızlı Çıktı

In [5]:
def quick_export(df_export, output_file_name, sheet_name=None, open_file=True):
    output_file_path = output_folder_path + output_file_name + '.xlsx'
    writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')
    if type(df_export) is list:
        if sheet_name is None:
            sheet_name = [x + 1 for x in range(len(df_export))]
        for df, sheet in zip(df_export, sheet_name):
            df.to_excel(writer, sheet_name = sheet, index = False)
    else:
        if sheet_name is None:
            sheet_name = output_file_name
        df_export.to_excel(writer, sheet_name = sheet_name, index = False)
    writer.close()
    if open_file:
        os.startfile(output_file_path)


    # def quick_export_csv(df, file_name, open_file=False):
#     df.to_csv(output_folder_path + file_name  + '.csv', sep=';', encoding='ANSI', index=False)

## Pickle

In [6]:
def save_pickle(file, name, sub=''):
    os.makedirs(pickle_folder_path + sub, exist_ok = True)
    with open(pickle_folder_path + sub + name, 'wb') as handle:
        pickle.dump(file, handle, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, sub=''):
    with open(pickle_folder_path + sub + name, 'rb') as handle:
        return pickle.load(handle)

# Mizan Test

In [7]:
mizan_test_file_name = test_file_name
mizan_test_pickle_name = 'mizan_df_list_prod'

xls = pd.ExcelFile(data_folder_path + mizan_test_file_name)
sheet_list = list(xls.sheet_names)
xls.close()

if os.path.exists(pickle_folder_path + mizan_test_pickle_name):
    mizan_df_list = load_pickle(mizan_test_pickle_name)
else:
    xls = pd.ExcelFile(data_folder_path + mizan_test_file_name)
    mizan_df_list = []
    for sheet in xls.sheet_names:
        print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
        df = pd.read_excel(xls, sheet_name=sheet)
        mizan_df_list.append(df)
    xls.close()
    save_pickle(mizan_df_list, mizan_test_pickle_name)

# Fix String

In [14]:
ssd = {
    'DM C': 'DMC',
    'D MC': 'DMC',
    'D M C': 'DMC',
    'SB': 'SUBE',

    'IST': 'ISTANBUL',
    'ANK': 'ANKARA',
    'IZM': 'IZMIR',

    'YURT ICI': 'YURTICI',
    'Y. ICI': 'YURTICI',
    'YURT DISI': 'YURTDISI',
    'Y. DISI': 'YURTDISI',
}

sd = {
    'AKADEMIK': 'AKADEMI',
    'AKA': 'AKADEMI',
    'AKAR': 'AKARYAKIT',
    'AKS': 'AKSESUAR',
    'ALIS': 'ALISVERIS',
    'ALS': 'ALISVERIS',
    'AMB': 'AMBALAJ',
    'AYAKKABICILIK': 'AYAKKABI',
    'AYAKKABICI': 'AYAKKABI',
    'AYAK': 'AYAKKABI',
    'BASIM EVI': 'BASIMEVI',
    'BAS': 'BASIM',
    'BELEDIYEBASKANLIGI': 'BELEDIYE BASKANLIGI',
    'BASK': 'BASKANLIGI',
    'BELGELENDIRME': 'BELGE',
    'BELG': 'BELGE',
    'BILGISAYARCILIK': 'BILGISAYAR',
    'BILG': 'BILGISAYAR',
    'BILIS': 'BILISIM',
    'BIL': 'BILISIM',
    'CEL': 'CELIK',
    'CIHAZLARI': 'CIHAZ',
    'CIHAZLAR': 'CIHAZ',
    'CIH': 'CIHAZ',
    'DAGIT': 'DAGITIM',
    'DAG': 'DAGITIM',
    'DG': 'DAGITIM',
    'DANISMAN': 'DANISMANLIK',
    'DANIS': 'DANISMANLIK',
    'DAN': 'DANISMANLIK',
    'DAY': 'DAYANIKLI',
    'DEGERLEME': 'DEGERLENDIRME',
    'DEG': 'DEGERLENDIRME',
    'DEKOR': 'DEKORASYON',
    'DEK': 'DEKORASYON',
    'DEN': 'DENETIM',
    'DERSHANECILIK': 'DERSHANE',
    'DERSANECILIK': 'DERSHANE',
    'DERSANE': 'DERSHANE',
    'DERSH': 'DERSHANE',
    'DERS': 'DERSHANE',
    'DER': 'DERSHANE',
    'EGIT': 'EGITIM',
    'EGTM': 'EGITIM',
    'EGT': 'EGITIM',
    'EG': 'EGITIM',
    'EKSP': 'EKSPERTIZ',
    'ELEKTRONIK': 'ELEKTRIK',
    'ELEKTR': 'ELEKTRIK',
    'ELEKT': 'ELEKTRIK',
    'ELEK': 'ELEKTRIK',
    'ELK': 'ELEKTRIK',
    'ENDUSTRIYEL': 'ENDUSTRI',
    'END': 'ENDUSTRI',
    'ESYALARI': 'ESYA',
    'ESYALAR': 'ESYA',
    'ESY': 'ESYA',
    'FABRIKALARI': 'FABRIKA',
    'FABRIKALAR': 'FABRIKA',
    'FABRIKASI': 'FABRIKA',
    'FAB': 'FABRIKA',
    'FLK': 'FLOK',
    'FOTOGRAF': 'FOTOGRAF',
    'FOTO': 'FOTOGRAF',
    'FOT': 'FOTOGRAF',
    'GAY': 'GAYRIMENKUL',
    'GEREC': 'GERECLERI',
    'GERC': 'GERECLERI',
    'GER': 'GERECLERI',
    'GID A': 'GIDA',
    'GID': 'GIDA',
    'GOZ': 'GOZETIM',
    'GUVENLIGI': 'GUVENLIK',
    'GUV': 'GUVENLIK',
    'HAYVAN': 'HAYVANCILIK',
    'HAY': 'HAYVANCILIK',
    'HZR': 'HAZIRLIK',
    'HAZ': 'HAZIRLIK',
    'HI HIZMETLERI': 'HIZMETLERI',
    'HIHIZMETLERI': 'HIZMETLERI',
    'HIZMETLE RI': 'HIZMETLERI',
    'HIZMET LERI': 'HIZMETLERI',
    'HIZMETLER': 'HIZMETLERI',
    'HIZM': 'HIZMETLERI',
    'HIZ': 'HIZMETLERI',
    'IHRC': 'IHRACAT',
    'IHR': 'IHRACAT',
    'ILETIS': 'ILETISIM',
    'ILET': 'ILETISIM',
    'ILE': 'ILETISIM',
    'IMALATI': 'IMALAT',
    'IML': 'IMALAT',
    'IM': 'IMALAT',
    'INS': 'INSAAT',
    'INT': 'INTERNASYONEL',
    'ISLETMELERI': 'ISLETME',
    'ISLETMECILIGI': 'ISLETME',
    'ISLETMECILIK': 'ISLETME',
    'ISLET': 'ISLETME',
    'ISLT': 'ISLETME',
    'ISL': 'ISLETME',
    'ITH': 'ITHALAT',
    'KERES': 'KERESTECILIK',
    'KIRA': 'KIRALAMA',
    'KIRTASI YE': 'KIRTASIYE',
    'KIRTASIY': 'KIRTASIYE',
    'KIRT': 'KIRTASIYE',
    'KIR': 'KIRTASIYE',
    'KERT': 'KIRTASIYE',
    'KIT': 'KITAP',
    'KONFEKS': 'KONFEKSIYON',
    'KONFEK': 'KONFEKSIYON',
    'KONF': 'KONFEKSIYON',
    'KONTR': 'KONTROL',
    'KONT': 'KONTROL',
    'KULT': 'KULTUR',
    'KUL': 'KULTUR',
    'KURUMLAR': 'KURUMLARI',
    'KURUM': 'KURUMLARI',
    'KUR': 'KURUMLARI',
    'KUY': 'KUYUMCULUK',
    'LABORATUAR': 'LABORATUVAR',
    'LAB': 'LABORATUVAR',
    'LOJ': 'LOJISTIK',
    'MADENCILIK': 'MADEN',
    'MADDE': 'MADDELERI',
    'MAD': 'MADDELERI',
    'MAGZ': 'MAGAZACILIK',
    'MAG': 'MAGAZACILIK',
    'MALZELERI': 'MALZEMELERI',
    'MALZE': 'MALZEMELERI',
    'MALZ': 'MALZEMELERI',
    'MLZ': 'MALZEMELERI',
    'MAL': 'MALZEMELERI',
    'MAKI NALARI': 'MAKINE',
    'MAKINALARI': 'MAKINE',
    'MAKINA': 'MAKINE',
    'MAK': 'MAKINE',
    'MAM': 'MAMULLERI',
    'MATB': 'MATBAACILIK',
    'MAT': 'MATBAACILIK',
    'MEK': 'MEKANIK',
    'MED': 'MEDYA',
    'MIMAR': 'MIMARLIK',
    'MIM': 'MIMARLIK',
    'MOB': 'MOBILYA',
    'MB': 'MOBILYA',
    'MOTORLU': 'MOTOR',
    'MOT': 'MOTOR',
    'MUDURLUK': 'MUDURLUGU',
    'MUD': 'MUDURLUGU',
    'MUHEN': 'MUHENDISLIK',
    'MUH': 'MUHENDISLIK',
    'MUSAVIRLIK': 'MUSAVIRLIGI',
    'MUSAVIR': 'MUSAVIRLIGI',
    'MUS': 'MUSAVIRLIGI',
    'NAKLIYECILIK': 'NAKLIYE',
    'NAKLIYAT': 'NAKLIYE',
    'NAKL': 'NAKLIYE',
    'NAK': 'NAKLIYE',
    'NES': 'NESRIYAT',
    'OGR': 'OGRETIM',
    'ORG': 'ORGANIZASYON',
    'ORT': 'ORTAKLIGI',
    'OTOMATIV': 'OTOMOTIV',
    'OTOM': 'OTOMOTIV',
    'OTO': 'OTOMOTIV',
    'OZ': 'OZEL',
    'P AZARLAMA': 'PAZARLAMA',
    'PAZARLAM': 'PAZARLAMA',
    'P AZ': 'PAZARLAMA',
    'PAZ': 'PAZARLAMA',
    'PZRL': 'PAZARLAMA',
    'PZR': 'PAZARLAMA',
    'PZ': 'PAZARLAMA',
    'PETR': 'PETROL',
    'PET': 'PETROL',
    'PNOM': 'PNOMATIK',
    'POL': 'POLIUERTAN',
    'PRO': 'PROJE',
    'REKL': 'REKLAMCILIK',
    'REK': 'REKLAMCILIK',
    'SAGLIGI': 'SAGLIK',
    'SAG': 'SAGLIK',
    'SERV': 'SERVIS',
    'SER': 'SERVIS',
    'SISTM': 'SISTEMLERI',
    'SIST': 'SISTEMLERI',
    'SI S': 'SISTEMLERI',
    'SIS': 'SISTEMLERI',
    'TAAHUT': 'TAAHHUT',
    'TAAH': 'TAAHHUT',
    'TAR': 'TARIM',
    'TASIMACILIGI': 'TASIMACILIK',
    'TASIMA': 'TASIMACILIK',
    'TAS': 'TASIMACILIK',
    'TEKNOLOJILERI': 'TEKNOLOJI',
    'TEKNOL': 'TEKNOLOJI',
    'TEKNO': 'TEKNOLOJI',
    'TENK': 'TEKNOLOJI',
    'TEKN': 'TEKNOLOJI',
    'TEK': 'TEKNOLOJI',
    'TEKS TIL': 'TEKSTIL',
    'TEKS': 'TEKSTIL',
    'TELEKOMUNUKASYON': 'TELEKOMUNIKASYON',
    'TELEKOMINIKASYON': 'TELEKOMUNIKASYON',
    'TELEKOM': 'TELEKOMUNIKASYON',
    'TELEK': 'TELEKOMUNIKASYON',
    'TEMIZ': 'TEMIZLIK',
    'TEM': 'TEMIZLIK',
    'TESISLER': 'TESISLERI',
    'TESIS': 'TESISLERI',
    'TES': 'TESISLERI',
    'TS': 'TESISLERI',
    'TI LT': 'TICARET',
    'TICLTD': 'TICARET',
    'UR': 'URETIM',
    'TIACERT': 'TICARET',
    'VETIC': 'TICARET',
    'TICARE T': 'TICARET',
    'TICAR': 'TICARET',
    'TICA': 'TICARET',
    'TIC': 'TICARET',
    'TI': 'TICARET',
    'TUK': 'TUKETIM',
    'TURZM': 'TURIZM',
    'TURZ': 'TURIZM',
    'TUR': 'TURIZM',
    'TRS': 'TURIZM',
    'TURISTIK': 'TURIZM',
    'URUNLERIVE': 'URUNLERI',
    'URUN': 'URUNLERI',
    'UR': 'URETIM',
    'UYGULAMALARI': 'UYGULAMA',
    'UYGULAMALAR': 'UYGULAMA',
    'UYGULAM': 'UYGULAMA',
    'UYG': 'UYGULAMA',
    'VETERINER': 'VETERINERLIK',
    'VETER': 'VETERINERLIK',
    'VET': 'VETERINERLIK',
    'YANG': 'YANGIN',
    'YAP': 'YAPI',
    'YAT': 'YATAK',
    'YAYIN': 'YAYINCILIK',
    'YAY': 'YAYINCILIK',
    'YAZI': 'YAZILIM',
    'YAZ': 'YAZILIM',
    'YONETIMI': 'YONETIM',
    'YON': 'YONETIM',
}

# sektor_list = ['AKADEMI', 'AKSESUAR', 'ALISVERIS', 'AMBALAJ', 'BASIMEVI', 'BASIM', 'BELEDIYE BASKANLIGI', 'BASKANLIGI', 'BELGE', 'BILGISAYAR', 'BILISIM', 'CIHAZLAR', 'DAGITIM', 'DANISMANLIK', 'DAYANIKLI', 'DEGERLENDIRME', 'DEKORASYON', 'DENETIM', 'DERSHANE', 'EGITIM', 'ELEKTRIK', 'ENDUSTRI', 'FABRIKASI', 'FOTOGRAF', 'GERECLERI', 'GIDA', 'GOZETIM', 'GUVENLIK', 'HAYVANCILIK', 'HAZIRLIK', 'HIZMETLERI', 'IHRACAT', 'ILETISIM', 'IMALAT', 'INSAAT', 'INTERNASYONEL', 'ISLETME', 'ITHALAT', 'KERESTECILIK', 'KIRTASIYE', 'KITAP', 'KONFEKSIYON', 'KONTROL', 'KULTUR', 'KURUMLARI', 'LABORATUVAR', 'LOJISTIK', 'MADDELERI', 'MAGAZACILIK', 'MALZEMELERI', 'MAKINE', 'MAMULLERI', 'MATBAACILIK', 'MEDYA', 'MIMARLIK', 'MOBILYA', 'MOTOR', 'MUHENDISLIK', 'MUSAVIRLIGI', 'NAKLIYE', 'NESRIYAT', 'OGRETIM', 'ORGANIZASYON', 'ORTAKLIGI', 'OTOMOTIV', 'OZEL', 'PAZARLAMA', 'PETROL', 'POLIUERTAN', 'PROMOSYON', 'REKLAMCILIK', 'SANAYI', 'SAGLIK', 'SERVIS', 'SISTEMLERI', 'TAAHHUT', 'TARIM', 'TASIMACILIK', 'TASARIM', 'TEKNOLOJI', 'TEKSTIL', 'TELEKOMUNIKASYON', 'TEMIZLIK', 'TESISLERI', 'TICARET', 'TUKETIM', 'TURIZM', 'URUNLERI', 'URETIM', 'UYGULAMA', 'VETERINERLIK', 'YANGIN', 'YAPI', 'YAYINCILIK', 'YAZILIM']

sektor_list = sorted(list(set([sd[x] for x in sd])))
sd = {**ssd, **sd}

In [15]:
def fix_string(x):
    sl = ['SANAYI', 'TICARET', 'TURIZM', 'DANISMANLIK', 'HIZMETLERI', 'ITHALAT', 'IHRACAT', 'IMALAT',
        'NAKLIYE', 'FABRIKASI', 'ENTEGRE', 'MOBILYA', 'LOJISTIK', 'PAZARLAMA', 'TEKSTIL', 'URUNLERI', 'INSAAT']
    
    y = x.upper()
    y += ' '
    y = y.replace('İ', 'I').replace('Ü', 'U').replace('Ö', 'O').replace('Ç', 'C').replace('Ş', 'S').replace('Ğ', 'G')
    
    y = y.replace(':', '.')
    y = y.replace('..', '.')
    y = y.replace('  ', ' ')
    y = y.replace('  ', ' ')

    for c in ['AR-GE', 'AR GE ']:
        y = y.replace(c, 'ARASTIRMA GELISTIRME')

    for c in ['.']:
        y = y.replace(c, '. ')
    
    for c in ['USD', 'EURO', 'EUR']:
        y = y.replace(f'-{c}', ' ')
        y = y.replace(f'({c})', ' ')

    for c in [
        ' VE ', 'VE.', 'V E ', ' HS. ', ' HS '
        '\xa0', '&AMP', '-', '/', '\\', '+', ',', '&',
        'SINIRLI SORUMLU ', 'YENI KURULACAK', 'TASFIYE HALINDE'
    ]:
        y = y.replace(c, ' ')

    for c in sl:
        y = y.replace(c, ' ' + c + ' ')
    
    for c in [
        ' SA ', 'SAN.', 'SAN ', 'SA N.', 'S AN.', 'SN.', 'SANL ', 'SANVE ',
        ' SANAYI I ', 'SANAYII', 'SANAY IVE', 'SANAYI IVE'
    ]:
        y = y.replace(c, ' SANAYI ')

    for c in ['T. A. S.']:
        y = y.replace(c, 'TURK ANONIM')
    
    for cc in [
        'SIRKE', 'SIRK', 'SIR', 'S TI', 'STI', 'ST', 'SI', 'S',
    ]:
        for c in [cc, cc + '.']:
            for i in range(3):
                if y.endswith(' ' + c + ' ' * i):
                    y = y[:-(len(c) + i + 1)]
    y += ' '

    for cc in [
        'LIMIT', 'LTD', 'LT D', 'LT  D', 'L TD', 'L  TD', 'LTI', 'LT', 'L', 
    ]:
        for c in [cc, cc + '.']:
            for i in range(3):
                if y.endswith(' ' + c + ' ' * i):
                    y = y.replace(c, 'LIMITED')
    y += ' '

    for cc in [
        'AN', 'ANO',
        'AS', 'A. S', 'A.  S', 'A . S.',
        'A S', 'A S.', 'A. S.', 'A', 
    ]:
        for c in [cc, cc + '.']:
            for i in range(3):
                if y.endswith(' ' + c + ' ' * i):
                    y = y.replace(c, 'ANONIM')
    y += ' '

    for c in [
        'LIMI TED', 'LIMITED', 'LIMITE', 'LTD', 'LT D' 'LIM', 'LTDSTI',
    ]:
        y = y.replace(f' {c} ',  'LIMITED ')
        y = y.replace(f' {c}.',  'LIMITED ')

    for c in [
        'A S', 'A  S', 'A. S', 'A.  S', 'ANOMIN', 'ANONI M', 'ANIONIM',
    ]:
        y = y.replace(f' {c} ',  'ANONIM ')
        y = y.replace(f' {c}.',  'ANONIM ')

    for c in sd:
        y = y.replace(f' {c} ', f' {sd[c]} ')
        y = y.replace(f' {c}.', f' {sd[c]} ')

    for c in sl:
        y = y.replace(c, ' ' + c + ' ')

    # for c in [' VE ', '-', ',', '&', 'SINIRLI SORUMLU ']:
    #     y = y.replace(c, ' ')

    y = y.replace('.', ' ')

    for i in range(5):
        y = y.replace('  ', ' ')
    return y.strip()


In [94]:
import re
x = 'ATLAS HARITA İNŞ.MÜ.PROJ.YAZ.DAN.TAAH.TIC.LTD.ŞTI.'
x.replace('.', ' ').replace('  ', ' ').split()

['ATLAS',
 'HARITA',
 'İNŞ',
 'MÜ',
 'PROJ',
 'YAZ',
 'DAN',
 'TAAH',
 'TIC',
 'LTD',
 'ŞTI']

In [16]:
fa = 'FA'
efa = 'EFA'
ifa = 'İFA'
iefa = 'İEFA'
iifa = 'İİFA'
sifa = 'SİFA'
ma = 'Müşteri Adı'
mn = 'Müşteri No'
vkn = 'VKN'

bs_codes = ['120', '320', '159', '340']
bank_codes = ['102', '300', '301', '302', '400', '401', '402']
account_code_column = 'ORJ_ACCOUNT_CODE'
account_name_column = 'ORJ_ACCOUNT_NAME'
bank_name_column = 'PROMETEIA_LABEL_BANK'
company_name_column = 'PROMETEIA_LABEL_CLIENT_NAME'
id_column = 'MIZAN_RAWDATA_ID'


In [25]:
def get_120_320(df):
    a = str(df[account_code_column])
    return any([a.startswith(x) for x in bs_codes]) and ('.' in a and not a.endswith('.0') and not a.endswith('.00'))

def get_bank(df):
    a = str(df[account_code_column])
    return any([a.startswith(x) for x in bank_codes]) and ('.' in a and not a.endswith('.0') and not a.endswith('.00'))

df_unmatched = pd.DataFrame(columns=['key', fa, ifa])
for df, sheet in zip(mizan_df_list, sheet_list):
    df = df.loc[(df[company_name_column].isnull()) & (df[account_name_column].notnull()) & (df[account_code_column].notnull())]
    df['120_320'] = df.apply(get_120_320, axis=1)
    df['key'] = sheet + '-' + df[id_column].astype(str)
    df = df.loc[df['120_320'], ['key', account_name_column, company_name_column]]
    df.columns = ['key', fa, ifa]
    df[ifa] = df[fa].apply(fix_string)
    df_unmatched = pd.concat([df_unmatched, df], ignore_index=True)
df_unmatched_backup = df_unmatched.copy()

df_correct = pd.DataFrame(columns=['key', ifa, iefa])
df_incorrect = pd.DataFrame(columns=['key', ifa, iefa])
for df, sheet in zip(mizan_df_list, sheet_list):
    df = df.loc[(df[company_name_column].notnull()) & (df[account_name_column].notnull()) & (df[account_code_column].notnull())]
    df['120_320'] = df.apply(get_120_320, axis=1)
    df['key'] = sheet + '-' + df[id_column].astype(str)
    df = df.loc[df['120_320'], ['key', account_name_column, company_name_column]]
    df.columns = ['key', fa, efa]
    df[ifa] = df[fa].apply(fix_string)
    df[iefa] = df[efa].apply(fix_string)
    dfc = df.loc[df[ifa] == df[iefa]]
    dfi = df.loc[df[ifa] != df[iefa]]
    df_correct = pd.concat([df_correct, dfc], ignore_index=True)
    df_incorrect = pd.concat([df_incorrect, dfi], ignore_index=True)

df_bank = pd.DataFrame(columns=['key'])
for df, sheet in zip(mizan_df_list, sheet_list):
    df = df.loc[(df[account_name_column].notnull()) & (df[account_code_column].notnull())]
    df['bank'] = df.apply(get_bank, axis=1)
    df['key'] = sheet + '-' + df[id_column].astype(str)
    df = df.loc[df['bank'], ['key']]
    df_bank = pd.concat([df_bank, df], ignore_index=True)

In [12]:
df = df_unmatched_backup.copy()
df['Öğrenci Şube'] = df[fa].apply(lambda x: any(y in x for y in ['OGRENCI', 'SUBE', 'OSS', 'SBS', 'LYS', 'YKS']))
df['Özel Karakter'] = df[fa].apply(lambda x: any(y in x for y in ':_-()/\\0123456789'))
df['2 Kelime'] = df[fa].apply(lambda x: len(x.replace('-', ' ').replace(':', ' ').replace('.', ' ').replace('/', ' ').split()) <= 2)
df['Gerçek Kişi'] = df.apply(lambda x: x['Öğrenci Şube'] or x['Özel Karakter'] or x['2 Kelime'], axis=1)

df_unmatched = df.copy()

In [13]:
overwrite = False
if os.path.exists(pickle_folder_path + 'df_findoc_backup'):
    dfm = load_pickle('df_findoc_backup')
else:
    df_findoc = pd.read_csv(base_path + 'ML\\FINDOC.csv', sep=';', encoding='ANSI')
    dfm = df_findoc.copy()
    dfm = dfm[['CLIENTID', 'FIRMNAME', 'TAXNO']]
    dfm.columns = [mn, ma, vkn]
    dfm = dfm.loc[(dfm[ma] != '#NAME?') & (dfm[ma].notnull()) & (dfm[mn].notnull()) & (dfm[vkn].notnull())]
    dfm = dfm.sort_values(mn).reset_index(drop=True)
    dfm = dfm.drop_duplicates(subset=[vkn], keep='first').reset_index(drop=True)
    dfm = dfm.drop_duplicates(subset=[ma], keep='last').reset_index(drop=True)
    dfm = dfm[[mn, ma]]
    save_pickle(dfm, 'df_findoc_backup')

if overwrite or not os.path.exists(pickle_folder_path + 'df_findoc_processed'):
    dfm = load_pickle('df_findoc_backup')
    dfm[ifa] = dfm[ma].apply(fix_string)
    dfm = dfm.drop_duplicates(subset=[ifa], keep='last').reset_index(drop=True)
    save_pickle(dfm, 'df_findoc_processed')
else:
    dfm = load_pickle('df_findoc_processed')

In [14]:
forbidden = ['E S', 'E', 'GIDA', 'ART', 'KAR', 'O KIMYA', 'E U', 'K D', 'SANAYI', 'SANAYI TICARET', 'AYAN', 'NOTER', 'AK BANK', ' CİVA LTD.'] + sektor_list

dfmm = dfm.copy()

dff = pd.merge(df_incorrect, dfmm, on=ifa, how='left')
dfif = dff.loc[dff['Müşteri No'].notnull()].copy().reset_index(drop=True)
dfinf = dff.loc[dff['Müşteri No'].isnull()].copy().reset_index(drop=True)

dff = pd.merge(df_unmatched, dfmm, on=ifa, how='left')
dfuf = dff.loc[dff['Müşteri No'].notnull()].copy().reset_index(drop=True)
dfusf = dff.loc[(dff['Müşteri No'].isnull()) & (dff['Gerçek Kişi'])].copy().reset_index(drop=True)
dfunf = dff.loc[(dff['Müşteri No'].isnull()) & (~dff['Gerçek Kişi'])].copy().reset_index(drop=True)


dfmm = dfmm.loc[~dfmm[ifa].isin(forbidden)]
dfmm = dfmm.sort_values('Müşteri No', ascending=False)
dfmm = dfmm.drop_duplicates(subset=[ifa], keep='first')
dfmm = dfmm.loc[dfmm[ifa].apply(lambda x: len(x) >= 4)]

In [105]:
dfa = pd.DataFrame(columns=['key', 'Sonuç', 'Müşteri No', 'Müşteri Adı'])

df = df_correct.copy()
df = df[['key']]
df['Sonuç'] = '1. Doğru Eşleşmiş'
dfa = pd.concat([dfa, df])

df = dfif.copy()
df = df[['key', 'Müşteri No', 'Müşteri Adı']]
df['Sonuç'] = '2. Yanlış Eşleşmiş - Bizde Var'
dfa = pd.concat([dfa, df])

df = dfinf.copy()
df = df[['key']]
df['Sonuç'] = '4. Yanlış Eşleşmiş - Bizde Yok'
dfa = pd.concat([dfa, df])

df = dfuf.copy()
df = df[['key', 'Müşteri No', 'Müşteri Adı']]
df['Sonuç'] = '3. Boş Gelmiş - Bizde Var'
dfa = pd.concat([dfa, df])

df = dfunf.copy()
df = df[['key']]
df['Sonuç'] = '6. Boş Gelmiş - Bizde Yok'
dfa = pd.concat([dfa, df])

df = dfusf.copy()
df = df[['key']]
df['Sonuç'] = '5. Boş Gelmiş - Gerçek Kişi'
dfa = pd.concat([dfa, df])

df = df_bank.copy()
df['Sonuç'] = '0. Banka'
dfa = pd.concat([dfa, df])

In [106]:
processed_mizan_list = []

for mizan, sheet in zip(mizan_df_list, sheet_list):
    df = mizan.copy()
    df['key'] = df.apply(lambda x: '{}-{}'.format(sheet, x[id_column]), axis=1)
    df = pd.merge(df, dfa, on='key', how='left')
    processed_mizan_list.append(df)


In [107]:
df_all = pd.concat(processed_mizan_list, ignore_index=True)

df_list = []
s_list = []

df = df_all.copy()
df = df.loc[df['Sonuç'] == '0. Banka'].reset_index(drop=True)
df = df.set_index('key').reset_index().drop([id_column, 'Sonuç'], axis=1)
df_list.append(df.copy())
s_list.append('Banka')

df = df_all.copy()
df = df.loc[df['Sonuç'] == '1. Doğru Eşleşmiş'].reset_index(drop=True)
df = df.set_index('key').reset_index().drop([id_column, 'Sonuç'], axis=1)
df_list.append(df.copy())
s_list.append('Doğru')

df = df_all.copy()
df = df.loc[df['Sonuç'] == '2. Yanlış Eşleşmiş - Bizde Var'].reset_index(drop=True)
df = df.set_index('key').reset_index().drop([id_column, 'Sonuç'], axis=1)
df_list.append(df.copy())
s_list.append('Yanlış')

df = df_all.copy()
df = df.loc[df['Sonuç'] == '3. Boş Gelmiş - Bizde Var'].reset_index(drop=True)
df = df.set_index('key').reset_index().drop([id_column, 'Sonuç'], axis=1)
df_list.append(df.copy())
s_list.append('Boş')

df = df_all.copy()
df = df.loc[df['Sonuç'] == '4. Yanlış Eşleşmiş - Bizde Yok'].reset_index(drop=True)
df = df.set_index('key').reset_index().drop([id_column, 'Sonuç'], axis=1)
df_list.append(df.copy())
s_list.append('Yanlış+')

df = df_all.copy()
df = df.loc[df['Sonuç'] == '5. Boş Gelmiş - Gerçek Kişi'].reset_index(drop=True)
df = df.set_index('key').reset_index().drop([id_column, 'Sonuç'], axis=1)
df_list.append(df.copy())
s_list.append('Gerçek')

df = df_all.copy()
df = df.loc[df['Sonuç'] == '6. Boş Gelmiş - Bizde Yok'].reset_index(drop=True)
df = df.set_index('key').reset_index().drop([id_column, 'Sonuç'], axis=1)
df_list.append(df.copy())
s_list.append('Boş+')


quick_export(df_list, 'Mizan Sonuç Prod 2', s_list)

# ARAMA

In [8]:
dfmm = load_pickle('dfmm')

In [115]:
name = 'ZEY SAM INSAAT DOGALGAZ'
strict = False

name = name.upper()
if ' ' in name:
    if strict:
        ddf = dfmm.loc[dfmm[ifa].apply(lambda x: all([' ' + n + ' ' in ' ' + x + ' ' for n in name.split()])), [mn, ma]].sort_values(ma).reset_index(drop=True)
    else:
        ddf = dfmm.loc[dfmm[ifa].apply(lambda x: all([n in x for n in name.split()])), [mn, ma]].sort_values(ma).reset_index(drop=True)
        
else:
    ddf = dfmm.loc[dfmm[ifa].apply(lambda x: ' ' + name + ' ' in ' ' + x + ' '), [mn, ma]].sort_values(ma).reset_index(drop=True)
# ddf = ddf.style.set_table_styles({
#     ma: [{'selector': '', 'props': [('width', '800px')]}],
# }, overwrite=False)
ddf = ddf.style.set_properties(subset=[ma], **{'text-align': 'left'})
ddf

,Müşteri No,Müşteri Adı
0,348684569,ZEY SAM İNŞAAT DOĞALGAZ TEKNİK TESİSAT GIDA TEMİZLİK TİCARET LİMİTED ŞİRKETİ


# Fuzzy

In [55]:
from collections import Counter

In [66]:
original = dfmm[ifa]
ls = [item for sublist in [x.split() for x in original] for item in sublist]
dc = Counter(ls)
df = pd.DataFrame()
df['Sektör'] = dc.keys()
df['Count'] = dc.values()

In [67]:
dx = df.sort_values('Count', ascending=False).reset_index(drop=True)
quick_export(dx, 'SEKTÖR')

In [50]:
[item.split() for item in ls]

[['KAVALA', 'TOPRAK'],
 ['S', 'S', 'TARIS', 'UZUM', 'TARIMSATIS', 'KOOPERATIFLERI', 'BIRLIGI'],
 ['TARIS', 'UZUM', 'ALKOLLU', 'ALKOLSUZ', 'ICECEKLER', 'SANAYI', 'TICARET'],
 ['ONAK', 'ZIRAAT', 'ILACLARITIC', 'SANAYI'],
 ['PAGMAT', 'PAMUK', 'TEKSTIL', 'GIDA', 'SANAYI', 'TICARET'],
 ['NOVAGRO', 'TARIM', 'URN', 'SANAYI', 'TICARET'],
 ['DOGAN',
  'AKARYAKIT',
  'TOPRAK',
  'MAH',
  'GIDA',
  'OTOMOTIV',
  'OTOMOTIV',
  'AL',
  'SAT',
  'DA'],
 ['MAKISSOS',
  'TARIM',
  'ILAC',
  'GUBRE',
  'TOHUM',
  'URETIM',
  'ITHALAT',
  'IHRACAT',
  'SANA'],
 ['ADAGRO', 'ZIRAI', 'ILAC', 'HAV', 'TARIM', 'SANAYI', 'TICARET'],
 ['AGROGLOB', 'TARIM', 'TARIMSAL', 'ILAC', 'SANAYI'],
 ['OSMAN',
  'ATICI',
  'NAKLIYE',
  'OTOMOTIV',
  'INSAAT',
  'SANAYI',
  'TICARET',
  'LIMI'],
 ['ONAK', 'ZIRAAT', 'ILACLARI'],
 ['ANTGEN',
  'KIMYA',
  'TARIM',
  'INSAAT',
  'TURIZM',
  'GIDA',
  'SANAYI',
  'TICARET',
  'LIM'],
 ['ACARLAR', 'ULUSLARARSI', 'DIS', 'TICARET'],
 ['ADA', 'DESING', 'YAPI', 'MALZEMELERI', 'TICARET

In [40]:
df_empty = pd.read_csv(data_folder_path + 'Tüzel Boş.csv', sep=';', encoding='ANSI')

In [ ]:
scope = 10
mn = 'Müşteri No'
fa = 'FA'
ifa = 'İFA'
ccs = ['key', fa, ifa]

dx = df_empty.copy()

matches_list = []
cstids_list = []
names = list(dx[ifa])
for i, name in enumerate(names):
    matches, scores, cstids = [], [], []
    for cstid, original, match in zip(dfmm[mn], dfmm[ma], dfmm[ifa]):
        score = fuzz.partial_ratio(name, match)
        matches.append(original)
        scores.append(score)
        cstids.append(cstid)
    matches = [x for _, x in sorted(zip(scores, matches), reverse=True)][:scope]
    cstids = [x for _, x in sorted(zip(scores, cstids), reverse=True)][:scope]
    matches_list.append(matches)
    cstids_list.append(cstids)
    print(f'{i + 1}/{len(names)} tamamlandı')

dx = dx[['key', fa]]
dx.columns = ['key', 'Mizan Firma Adı']
for i in range(scope):
    dx['MüşteriNo' + str(i+1)] = [x[i] for x in cstids_list]
    dx['Eşleşme' + str(i+1)] = [x[i] for x in matches_list]

quick_export(dx, 'Tüzel Boş Fuzzy')